In [3]:
import gradio as gr

In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [5]:
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

In [6]:
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

Google API Key exists and begins AIzaSyAs


In [7]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [8]:
# Let's wrap a call to the Gemini API in a simple function

system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, 
                {"role": "user", "content": prompt}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)
    return response.choices[0].message.content

In [ ]:
message_gpt("What is today's date?")

In [9]:
# here's a simple function

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [ ]:
shout("hello")

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("ed", "bananas"))

In [ ]:
# Define this variable and then pass js=force_dark_mode when creating the Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:

message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout",  
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Gemini", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
system_message = "You are smart assistant that responds in markdown without any code blocks"
message_input = gr.Textbox(label="Your message:", info="Enter a message for Gemini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture in detail",
        "Write a haiku about the beach"], 
    flagging_mode="never"
    )
view.launch()

In [10]:
 ## Let's create a call that streams back results
# refresher on Generators (the "yield" keyword),
# Please take a look at the Intermediate Python guide in the guides folder

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = gemini.chat.completions.create(
        model='gemini-2.5-flash-lite',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [25]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

In [26]:
openai = OpenAI(base_url = OLLAMA_BASE_URL, api_key="ollama")

In [27]:
def stream_llama(prompt):
    messages = [
        {"role": "system", "content": system_message},      
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model= "llama3.2",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result
    

In [29]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [28]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Llama", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_llama,
    title="Llama", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [30]:
## now using gemini and llama models!
def stream_model(prompt,model):
    if model == "Gemini":
        result = stream_gpt(prompt)
    elif model == "Llama":        
        result = stream_llama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["Gemini", "Llama"], label="Select model", value="Gemini")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "Gemini"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Llama"]
        ], 
    flagging_mode="never"
    )
view.launch()

e:\LLM\venv\Lib\site-packages\gradio\components\dropdown.py:235: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: GPT or set allow_custom_value=True.
  warnings.warn(
e:\LLM\venv\Lib\site-packages\gradio\components\dropdown.py:235: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Claude or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
